# Monitor, troubleshoot and optimize workloads in Azure Databricks

## Monitor and manage cluster consumption

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/monitor-troubleshoot-optimize-workloads-azure-databricks.monitor-manage-cluster-consumption.png)

### Understand the impact of cluster consumption

> **Trainer Demo Guide**
>
> 1. Navigate to **Account Console** > **Usage** to see overall DBU usage trends
> 2. Point out examples of idle clusters (flat lines in the chart between work hours)
> 3. Show a weekend or overnight usage spike as an example of unmonitored costs
>
> **Key message:** The goal is to **match resources precisely to workload requirements** -- not simply minimise costs.

---

### Monitor compute metrics

> **Trainer Demo Guide**
>
> 1. Go to **Compute** in the sidebar, select your cluster, click the **Metrics** tab
> 2. Walk through the three metric categories:
>    - **Hardware metrics**: CPU utilization, memory utilization, network throughput
>    - **Spark metrics**: active tasks, failed/completed tasks, shuffle read/write
>    - **GPU metrics** (available on Databricks Runtime ML 13.3+)
> 3. Use the date picker to filter data from the last 30 days
> 4. Select individual nodes in the **Compute** dropdown to compare worker performance
>
> **Key signal:** High shuffle read/write values indicate data movement bottlenecks worth investigating.
>
> Run the code cell below to inspect recent cluster metadata from system tables.

In [ ]:
# === DEMO: Inspect recent cluster metadata from system tables ===
# system.compute.clusters contains metadata about all clusters in your account

spark.sql("""
    SELECT
        cluster_id,
        cluster_name,
        cluster_source,
        owned_by,
        state,
        start_time,
        terminated_time
    FROM system.compute.clusters
    ORDER BY start_time DESC
    LIMIT 20
""").display()

### Monitor SQL warehouse performance

> **Trainer Demo Guide**
>
> 1. Go to **SQL Warehouses** in the left sidebar
> 2. Select a warehouse and open the **Monitoring** tab
> 3. Walk through the live statistics panel:
>    - **Status**, **Running queries**, **Queued queries**, **Cluster count**
> 4. Show the **Peak query count** chart -- spikes indicate periods of high demand
> 5. Show the **Running clusters** chart -- autoscaling behaviour in action
> 6. In **Query history**, filter for long-running queries and discuss resource impact
>
> **Key points:**
> - If the cluster count stays at maximum for extended periods, increase the max cluster limit
> - Use query history to identify users or queries consuming disproportionate resources

### Configure auto-termination and autoscaling

> **Trainer Demo Guide**
>
> **Auto-termination:**
> 1. Go to **Compute** > Edit an all-purpose cluster
> 2. Enable **Auto termination** and set to **30-60 minutes** for development clusters
> 3. The cluster only terminates when *all* activity stops (Spark jobs, Structured Streaming, JDBC calls)
>
> **Autoscaling:**
> 1. Enable the **Autoscaling** toggle and set minimum (e.g., 2) and maximum (e.g., 8) worker nodes
> 2. Databricks auto-adjusts -- typical savings of **20-40%** vs. fixed-size clusters
> 3. Contrast with serverless compute: serverless scales to zero automatically
>
> **Important:** Auto-termination does NOT monitor DStreams. If using DStream workloads, disable auto-termination or migrate to Structured Streaming.

### Track costs with budgets and system tables

> **Trainer Demo Guide**
>
> **Budgets (Account Console):**
> 1. Navigate to **Account Console** > **Usage** > **Budgets**
> 2. Click **Add budget**, configure a spend limit, and set up email alerts
> 3. Requires **account admin** permissions
>
> **System tables for cost analysis:**
> - `system.billing.usage` -- DBU consumption per cluster, job, and SKU
> - `system.compute.clusters` -- cluster metadata including owner
> - Join these tables to understand who is spending what
>
> **Tip:** Apply tags (e.g., `business_unit`, `project`, `environment`) to clusters and workspaces from the start -- they propagate to billing records and enable accurate chargeback.
>
> Run the SQL queries below to demonstrate cost analysis in action.

In [ ]:
# === DEMO: Daily DBU consumption trend ===
# See how compute usage changes day by day -- spot unusual spikes or weekly patterns

spark.sql("""
    SELECT
        usage_date,
        SUM(usage_quantity) AS DBUs_Consumed
    FROM system.billing.usage
    WHERE sku_name = 'STANDARD_ALL_PURPOSE_COMPUTE'
    GROUP BY usage_date
    ORDER BY usage_date ASC
""").display()

In [ ]:
# === DEMO: Which jobs consumed the most DBUs? ===
# Identify the biggest cost drivers among scheduled jobs

spark.sql("""
    SELECT
        usage_metadata.job_id AS job_id,
        SUM(usage_quantity)   AS total_dbus
    FROM system.billing.usage
    WHERE usage_metadata.job_id IS NOT NULL
    GROUP BY job_id
    ORDER BY total_dbus DESC
    LIMIT 20
""").display()

In [ ]:
# === DEMO: Attribute costs to cluster owners ===
# Identify which team members or service accounts drive the most resource consumption

spark.sql("""
    SELECT
        c.owned_by,
        SUM(u.usage_quantity) AS total_dbus
    FROM system.billing.usage u
    JOIN system.compute.clusters c
        ON u.usage_metadata.cluster_id = c.cluster_id
    WHERE u.usage_metadata.cluster_id IS NOT NULL
    GROUP BY c.owned_by
    ORDER BY total_dbus DESC
""").display()

In [ ]:
# === DEMO: Usage by custom tag ===
# Break down costs by business unit, project, or environment tag

spark.sql("""
    SELECT
        sku_name,
        usage_unit,
        custom_tags['project'] AS project_tag,
        SUM(usage_quantity)    AS total_usage
    FROM system.billing.usage
    WHERE custom_tags['project'] IS NOT NULL
    GROUP BY sku_name, usage_unit, custom_tags['project']
    ORDER BY total_usage DESC
""").display()

## Troubleshoot and repair Lakeflow Jobs

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/monitor-troubleshoot-optimize-workloads-azure-databricks.troubleshoot-repair-lakeflow-jobs.png)

### Understand job failure states

When a Lakeflow Job run ends, it can be in one of these states:

| State | Meaning |
|-------|---------|
| **Failed** | One or more leaf tasks didn't complete successfully |
| **Succeeded with failures** | Some tasks failed, but all leaf tasks completed |
| **Skipped** | Job exceeded maximum concurrent runs |
| **Timed Out** | Job exceeded configured maximum duration |
| **Canceled** | Stopped by user or automated process |

---

### Identify the cause of failure

> **Trainer Demo Guide**
>
> 1. Navigate to **Jobs & Pipelines** in the sidebar
> 2. Select a job and open the **Runs** tab
> 3. Show the **matrix view** -- each cell is a task status across multiple runs
> 4. Hover over a red (failed) task: see start time, end time, status, error message
> 5. Click the failed task to open **Task run details** with full output and logs
> 6. Click **Diagnose Error** to invoke Databricks Assistant for suggestions
>
> **Pattern recognition:**
> - Same task fails repeatedly -> code or configuration issue specific to that task
> - Random failures across different tasks -> cluster or resource problem

In [ ]:
# === DEMO: Simulate a task-level data quality failure ===
# Demonstrates how schema/type issues surface and how to detect them

from pyspark.sql.functions import col

# Dataset with a mixed-type 'amount' column
data = [
    (1, 'Alice', '150.00'),     # valid
    (2, 'Bob',   'not_a_num'),  # invalid -- cast produces NULL by default
    (3, 'Carol', '275.50'),     # valid
    (4, 'Dave',  None),         # null
]
df = spark.createDataFrame(data, ['id', 'name', 'amount_str'])

# Default: invalid casts produce NULL (no exception)
df_cast = df.withColumn('amount', col('amount_str').cast('double'))
df_cast.show()

# Detect rows that failed the cast (practical quarantine pattern)
bad_rows = df_cast.filter(col('amount').isNull() & col('amount_str').isNotNull())
print(f'Rows failed cast: {bad_rows.count()}')
bad_rows.show()

# To raise a hard failure visible in the Spark UI, enable ANSI mode:
# spark.conf.set('spark.sql.ansi.enabled', 'true')
# spark.sql("SELECT CAST('not_a_num' AS DOUBLE)").show()

### Repair failed job runs

> **Trainer Demo Guide**
>
> 1. In the **Runs** tab, click the failed run in the **Start time** column
> 2. Click **Repair run** in the top-right corner
> 3. Review the list of tasks that will re-run (failed tasks + their dependents)
> 4. Optionally modify task parameters in the repair dialog
> 5. Click **Repair run** to start the re-execution
>
> **Key points:**
> - Only failed tasks and their dependents re-run -- saves time and compute costs
> - The matrix view adds a new column after repair completes
> - Repair is only available for jobs with **2 or more tasks**
>
> **Stop / Restart controls:**
> - **Stop**: click the stop button on an active run in the Runs tab
> - **Restart continuous jobs**: use **Restart run** to bypass exponential backoff
>   after repeated failures

In [ ]:
# === DEMO: Analyze Lakeflow job run history with system tables ===
# system.lakeflow.job_runs provides programmatic access to job execution data

spark.sql("""
    SELECT
        job_id,
        run_id,
        run_type,
        result_state,
        start_time,
        end_time,
        TIMESTAMPDIFF(MINUTE, start_time, end_time) AS duration_minutes
    FROM system.lakeflow.job_runs
    ORDER BY start_time DESC
    LIMIT 20
""").display()

### Monitor job health proactively

> **Trainer Demo Guide**
>
> **Configure job notifications:**
> 1. Open a job > click **Edit** > scroll to **Notifications**
> 2. Add email, Slack, or webhook alerts for: start, success, failure, duration exceeded
> 3. Show where to configure **Maximum retries** and **Retry interval**
>
> **Best practices:**
> - Review the **matrix view** regularly -- patterns of failure in specific tasks indicate areas needing attention
> - Use `system.lakeflow` tables to build cross-workspace dashboards for ops teams
> - Set a **maximum duration** on long-running jobs to prevent runaway costs

## Troubleshoot Spark jobs and notebooks

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/monitor-troubleshoot-optimize-workloads-azure-databricks.troubleshoot-repair-spark-jobs-notebooks.png)

### Understand common causes of Spark job failures

Spark job failures typically fall into three categories:

| Category | Examples | First Signal |
|----------|---------|--------------|
| **Code errors** | Syntax errors, schema mismatches, data quality issues | Error message with stack trace |
| **Resource bottlenecks** | Out-of-memory (OOM), slow shuffle, spill to disk | Long task duration, executor failures |
| **Environmental issues** | Cluster startup failure, spot instance reclamation | Event log entries |

---

### Use the Spark UI to diagnose issues

> **Trainer Demo Guide**
>
> 1. Navigate to your cluster > **Spark UI** tab > **Jobs** page
> 2. Look at the **Jobs Timeline** for three patterns:
>    - **Failing jobs**: red status -- click to see the failed stage and error
>    - **Long-running jobs**: single job dominating the timeline -- check its stages
>    - **Gaps in execution**: no Spark work happening -- driver may be overloaded
> 3. Click into a long-running job > **Stages** > identify the longest stage
> 4. In the stage detail, examine **Summary Metrics**:
>    - **Input/Output**: data read/written
>    - **Shuffle Read/Write**: data moved between nodes
>    - **Task duration distribution**: look for outlier tasks (data skew)
>
> **Key insight:** A stage with only 1 task = insufficient parallelism. Uneven task durations = data skew.
>
> Run the code cell below to generate a multi-stage Spark job to examine in the Spark UI.

In [ ]:
# === DEMO: Generate a multi-stage Spark job to analyze in the Spark UI ===
# Creates visible stages, shuffle, and aggregation patterns

from pyspark.sql.functions import col, sum as spark_sum, count

print('Generating data for Spark UI analysis...')

# Stage 1: create base data
df = (spark.range(0, 500_000)
        .toDF('id')
        .withColumn('category', (col('id') % 100).cast('string'))
        .withColumn('value',    (col('id') * 3.14).cast('double')))

# Stage 2: aggregation triggers a shuffle (sort-merge or hash aggregate)
agg_df = df.groupBy('category').agg(
    count('*').alias('record_count'),
    spark_sum('value').alias('total_value')
)

# Stage 3: sort triggers another shuffle
result = agg_df.orderBy('total_value', ascending=False)
result.display()

print('\nJob complete.')
print('Navigate to: Compute > Your Cluster > Spark UI > Jobs')
print('Examine the three stages, shuffle read/write sizes, and task distribution.')

### Identify and resolve resource bottlenecks

> **Trainer Demo Guide** -- Spark UI: Metrics tab
>
> **Memory pressure:**
> - High memory utilization + **Shuffle Spill (Disk) > 0** in stage metrics
> - Fix: increase worker instance sizes, reduce partition count, optimize transformations
>
> **CPU constraints:**
> - High CPU utilization with long task execution times
> - Fix: enable **Photon** for compatible workloads, add worker nodes
>
> **Network bottlenecks:**
> - Large **Shuffle Read/Write** volumes with slow execution
> - Fix: filter data earlier, use broadcast joins, reduce unnecessary repartitions
>
> **Server load distribution (Metrics tab):**
> - Red = heavily loaded, Blue = idle
> - Driver in red + workers in blue -> driver is overloaded, needs a larger instance type
> - Navigate to: **Compute** > Your Cluster > **Metrics** > Server load distribution

In [ ]:
# === DEMO: Inspect cluster termination events from system tables ===
# Look for abnormal termination codes indicating resource or environmental issues

spark.sql("""
    SELECT
        cluster_id,
        cluster_name,
        owned_by,
        state,
        start_time,
        terminated_time,
        termination_code,
        termination_type
    FROM system.compute.clusters
    WHERE termination_code IS NOT NULL
      AND termination_code != 'USER_REQUEST'
    ORDER BY terminated_time DESC
    LIMIT 20
""").display()

### Restart clusters to resolve environmental issues

> **Trainer Demo Guide**
>
> **Before restarting, check the Event log:**
> 1. **Compute** > select your cluster > **Event log** tab
> 2. Look for: `DRIVER_NOT_RESPONDING`, spot instance reclamation, executor terminations
>
> **Restart via UI:**
> 1. **Compute** > select your cluster
> 2. Click **Restart** and confirm -- all running jobs are terminated
>
> **Restart via Databricks CLI:**
> ```bash
> # List clusters to find your CLUSTER_ID
> databricks clusters list
>
> # Restart the cluster
> databricks clusters restart --cluster-id <CLUSTER_ID>
> ```
>
> **Important:** Restarting terminates running jobs and resets the Spark UI history. Save diagnostic information before restarting.

## Investigate caching, skewing, spilling, shuffle

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/monitor-troubleshoot-optimize-workloads-azure-databricks.resolve-caching-skewing-spilling-shuffle.png)

### Use diagnostic tools to identify issues

Azure Databricks provides three primary tools for investigating performance problems:

| Tool | Best for | Access |
|------|----------|--------|
| **Query DAG** | Understanding execution flow, operator bottlenecks | Spark UI > SQL/DataFrame tab |
| **Spark UI** | Stage/task metrics, shuffle volumes, spill, task skew | Compute > Cluster > Spark UI |
| **Query Profile** | SQL warehouse analysis, per-operator timing | SQL Editor > Query History > See query profile |

> **Tip:** Start with the DAG to understand the overall flow, then use Spark UI or query profile to drill into specific slow stages.

In [ ]:
# === DEMO: Check disk cache (Delta cache) and Spark cache settings ===

# 1. Check if disk cache is enabled (automatically caches Parquet files on worker SSDs)
disk_cache = spark.conf.get('spark.databricks.io.cache.enabled')
print(f'Disk cache (Delta cache) enabled: {disk_cache}')

# 2. Demonstrate Spark in-memory cache with .cache() and .unpersist()
from pyspark.sql.functions import col
import time

df = (spark.range(0, 500_000)
        .withColumn('value',    (col('id') * 2.5).cast('double'))
        .withColumn('category', (col('id') % 20).cast('string')))

# Cache the DataFrame in memory
df.cache()

# First action: materializes and caches
t0 = time.time(); df.count(); t1 = time.time()
print(f'First count  (cache miss): {t1 - t0:.2f}s')

# Second action: served from cache -- should be faster
t0 = time.time(); df.count(); t1 = time.time()
print(f'Second count (cache hit):  {t1 - t0:.2f}s')

# IMPORTANT: free memory when done
df.unpersist()
print('Cache cleared with .unpersist()')
print('\nCheck Spark UI > Storage tab to see currently cached DataFrames.')

In [ ]:
# === DEMO: Investigate data skew ===
# Skew = some partitions contain far more data than others
# Symptom: a few tasks take much longer than the rest in the same stage

from pyspark.sql import Row
from pyspark.sql.functions import col, count

# Create a SKEWED dataset: 90% of records have category 'A'
skewed_data = (
    [Row(id=i, category='A')       for i in range(90_000)] +  # 90% skewed
    [Row(id=i, category=str(i%9))  for i in range(10_000)]    # 10% distributed
)
skewed_df = spark.createDataFrame(skewed_data)

print('Running aggregation on skewed data...')
print('Open Spark UI > Stages > Summary Metrics while this runs.')
print('Max task duration >> 75th percentile = skew detected\n')

result = skewed_df.groupBy('category').count().orderBy('count', ascending=False)
result.show()

print('\nRemedies for data skew:')
print('  1. Enable AQE (see next cell) -- handles skewed joins automatically')
print('  2. Salt skewed keys: add a random prefix, aggregate in two steps')
print('  3. Broadcast the smaller table in a join to avoid shuffle-based skew')

In [ ]:
# === DEMO: Adaptive Query Execution (AQE) ===
# AQE dynamically optimises queries at runtime, including skewed joins

# Check AQE status
aqe_db = spark.conf.get('spark.databricks.optimizer.adaptive.enabled')
aqe_sp = spark.conf.get('spark.sql.adaptive.enabled')
print(f'spark.databricks.optimizer.adaptive.enabled: {aqe_db}')
print(f'spark.sql.adaptive.enabled:                  {aqe_sp}')

print("""
AQE handles three key optimisations automatically:
  1. Skewed join optimisation  -- detects and splits large partitions at runtime
  2. Dynamic partition coalescing -- merges small shuffle partitions after a stage
  3. Join strategy switch -- converts sort-merge joins to broadcast joins when
     a table turns out to be small enough during execution

In the Spark UI (SQL/DataFrame tab), AQE-optimised plans show:
  'AQEShuffleRead' and 'CustomShuffleReader' nodes
""")

In [ ]:
# === DEMO: Memory spill -- investigation and prevention ===

# Check current shuffle partition setting
current = spark.conf.get('spark.sql.shuffle.partitions')
print(f'Current spark.sql.shuffle.partitions: {current}')

# Set to 'auto' for AQE-driven dynamic optimisation
spark.conf.set('spark.sql.shuffle.partitions', 'auto')
print(f'After set to auto:                    {spark.conf.get("spark.sql.shuffle.partitions")}')

print("""
To investigate spill in the Spark UI:
  1. Spark UI > Stages > click a stage involving aggregation or sort
  2. At the top of the stage page look for:
       Shuffle Spill (Memory) -- data spilled from RAM to off-heap
       Shuffle Spill (Disk)   -- data written to disk due to memory pressure
  3. Non-zero values indicate memory pressure

Remedies for spill:
  - Increase partition count (spark.sql.shuffle.partitions)
  - Use workers with higher memory-to-core ratios
  - Select only needed columns early (column pruning) to reduce data volume
  - Fix data skew first -- skew and spill often go together
""")

In [ ]:
# === DEMO: Shuffle investigation and broadcast join optimisation ===
# Shuffle moves data between nodes during joins, aggregations, repartitioning

from pyspark.sql.functions import broadcast, col

# Large fact table and small dimension table
large_df = (spark.range(0, 500_000)
              .toDF('id')
              .withColumn('category', (col('id') % 50).cast('string')))

small_df = spark.createDataFrame(
    [(str(i), f'Category-{i}') for i in range(50)],
    ['category', 'category_name']
)

print('=== WITHOUT broadcast hint -- sort-merge join (causes shuffle) ===')
no_broadcast = large_df.join(small_df, on='category', how='left')
no_broadcast.explain(mode='simple')

print('\n=== WITH broadcast hint -- broadcast hash join (no shuffle) ===')
with_broadcast = large_df.join(broadcast(small_df), on='category', how='left')
with_broadcast.explain(mode='simple')

# Execute the broadcast version
with_broadcast.count()
print('\nCompare the query plans above.')
print('Look for SortMergeJoin (shuffle) vs. BroadcastHashJoin (no shuffle).')
print('Spark UI > SQL/DataFrame > DAG: Exchange nodes indicate shuffle.')

## Implement log streaming with Azure Log Analytics

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/monitor-troubleshoot-optimize-workloads-azure-databricks.implement-log-streaming-azure-analytics.png)

### Understand the log streaming architecture

> **Trainer Demo Guide** -- one-time setup (platform admin)
>
> 1. Open **Azure portal** > navigate to your Azure Databricks workspace resource
> 2. Under **Monitoring**, click **Diagnostic settings**
> 3. Click **+ Add diagnostic setting**
> 4. Select log categories: `clusters`, `jobs`, `notebook`, `SQL`, `workspace`, `unityCatalog`
> 5. Set destination to your **Log Analytics workspace**
> 6. Save -- logs appear in Log Analytics within approximately 15 minutes
>
> **Data flow:**
> ```
> Azure Databricks events
>   --> Azure Diagnostic Settings
>     --> Log Analytics workspace (service-specific tables)
>       --> KQL queries / alerts / dashboards
> ```
>
> **Prerequisites:**
> - Azure Databricks **Premium plan** (required for diagnostic logs)
> - A Log Analytics workspace in Azure
>
> **What gets captured:** cluster lifecycle, job runs/failures, notebook executions,
> Unity Catalog data access, secret access events

### Explore the Azure Databricks log tables

Log Analytics organises Azure Databricks logs into purpose-specific tables:

| Table | Description |
|-------|-------------|
| `DatabricksClusters` | Cluster creation, termination, resizing, configuration changes |
| `DatabricksJobs` | Job creation, runs, failures, schedule modifications |
| `DatabricksNotebook` | Notebook execution, creation, and modification events |
| `DatabricksSQL` | SQL warehouse operations and query events |
| `DatabricksUnityCatalog` | Catalog, schema, and table access events |
| `DatabricksWorkspace` | Workspace-level and administrative actions |
| `DatabricksSecrets` | Secret scope and secret access events |

**Standard columns in every table:**
- `TimeGenerated` -- when the event occurred
- `OperationName` -- the action performed
- `Identity` -- the user or service principal
- `SourceIPAddress` -- where the request originated
- `RequestParams` -- details about the request
- `Response` -- outcome including status codes

> **Tip:** Save frequently used queries to the **Query explorer** in Log Analytics. Organise them in folders and share with your team.

### Query logs with Kusto Query Language (KQL)

In the **Azure portal**, navigate to your Log Analytics workspace and open **Logs**.

**Recent job events (last 24 hours):**
```kusto
DatabricksJobs
| where TimeGenerated > ago(24h)
| project TimeGenerated, OperationName, Identity, SourceIPAddress
| order by TimeGenerated desc
| take 100
```

**Identify job failure patterns:**
```kusto
DatabricksJobs
| where TimeGenerated > ago(7d)
| where Response contains "error" or Response contains "failed"
| summarize FailureCount = count() by OperationName, bin(TimeGenerated, 1h)
| order by FailureCount desc
```

**Track cluster usage events:**
```kusto
DatabricksClusters
| where TimeGenerated > ago(7d)
| summarize EventCount = count() by OperationName
| order by EventCount desc
```

**Monitor Unity Catalog access (security and compliance):**
```kusto
DatabricksUnityCatalog
| where TimeGenerated > ago(24h)
| where OperationName has "getTable" or OperationName has "selectFromTable"
| project TimeGenerated, Identity, OperationName, RequestParams
| order by TimeGenerated desc
```

---

### Create alerts for proactive monitoring

> **Trainer Demo Guide**
>
> 1. Run your KQL query in Log Analytics
> 2. Click **New alert rule** in the toolbar
> 3. Configure the **Condition** (e.g., FailureCount > 5 per hour)
> 4. Define an **Action group**: email, SMS, webhook (Teams/Slack)
> 5. Set Alert rule name, severity, and review frequency
> 6. Click **Review + create**
>
> **Use case examples:**
> - Alert when job failure rate exceeds a threshold
> - Notify on unusual data access patterns in Unity Catalog
> - Trigger a webhook when a critical cluster terminates unexpectedly